In [18]:
import openmeteo_requests
import requests_cache
import pandas as pd
from retry_requests import retry

# Setup the Open-Meteo API client with cache and retry on error
cache_session = requests_cache.CachedSession('.cache', expire_after=3600)
retry_session = retry(cache_session, retries=5, backoff_factor=0.2)
openmeteo = openmeteo_requests.Client(session=retry_session)

# List of cities with their latitudes and longitudes
cities = {
    "Delhi": (28.679079, 77.069710),
    "Mumbai": (19.07283, 72.88261),
    "Kolkata": (22.54111111, 88.33777778),
    "Bangalore": (12.9767936, 77.5900820),
    "Hyderabad": (17.38405000, 78.45636000),
    "Chennai": (13.08784, 80.27847),
    "Ahmedabad": (23.02579000, 72.58727000),
    "Surat": (21.170240, 72.831062),
    "Pune": (18.51957000, 73.85535000),
    "Jaipur": (26.907524, 75.739639),
    "Lucknow": (26.83928000, 80.92313000),
    "Kanpur": (26.46523000, 80.34975000),
    "Nagpur": (21.14631000, 79.08491000),
    "Visakhapatnam": (17.68009000, 83.20161000),
    "Bhopal": (23.25469000, 77.40289000),
    "Indore": (22.66667000, 75.75000000),
    "Thane": (19.1967, 72.9636),
    "Ghaziabad": (28.6489, 77.4422),
    "Noida": (28.5355, 77.3910),
    "Coimbatore": (11.0018, 76.9628)
}

# Parameters for the API request
params = {
    "hourly": ["temperature_2m", "relativehumidity_2m", "windspeed_10m"],
    "daily": ["temperature_2m_max", "temperature_2m_min"],
    "past_days": 31,
    "forecast_days": 1
}

# Function to fetch and process weather data for a city
def fetch_and_process_weather_data(city, lat, lon):
    city_params = params.copy()
    city_params["latitude"] = lat
    city_params["longitude"] = lon

    responses = openmeteo.weather_api("https://api.open-meteo.com/v1/forecast", params=city_params)
    response = responses[0]

    print(f"Processing data for {city}...")
    print(f"Coordinates {response.Latitude()}°N {response.Longitude()}°E")
    print(f"Elevation {response.Elevation()} m asl")
    print(f"Timezone {response.Timezone()} {response.TimezoneAbbreviation()}")
    print(f"Timezone difference to GMT+0 {response.UtcOffsetSeconds()} s")

    # Process hourly data
    hourly = response.Hourly()
    hourly_temperature_2m = hourly.Variables(0).ValuesAsNumpy()
    hourly_relative_humidity_2m = hourly.Variables(1).ValuesAsNumpy()
    hourly_wind_speed_10m = hourly.Variables(2).ValuesAsNumpy()

    hourly_data = {
        "date": pd.date_range(
            start=pd.to_datetime(hourly.Time(), unit="s", utc=True),
            end=pd.to_datetime(hourly.TimeEnd(), unit="s", utc=True),
            freq=pd.Timedelta(seconds=hourly.Interval()),
            inclusive="left"
        ),
        "temperature_2m": hourly_temperature_2m,
        "relative_humidity_2m": hourly_relative_humidity_2m,
        "wind_speed_10m": hourly_wind_speed_10m
    }

    hourly_dataframe = pd.DataFrame(data=hourly_data)
    hourly_dataframe['City'] = city

    # Process daily data
    daily = response.Daily()
    daily_temperature_2m_max = daily.Variables(0).ValuesAsNumpy()
    daily_temperature_2m_min = daily.Variables(1).ValuesAsNumpy()

    daily_data = {
        "date": pd.date_range(
            start=pd.to_datetime(daily.Time(), unit="s", utc=True),
            end=pd.to_datetime(daily.TimeEnd(), unit="s", utc=True),
            freq=pd.Timedelta(seconds=daily.Interval()),
            inclusive="left"
        ),
        "temperature_2m_max": daily_temperature_2m_max,
        "temperature_2m_min": daily_temperature_2m_min
    }

    daily_dataframe = pd.DataFrame(data=daily_data)
    daily_dataframe['City'] = city

    return hourly_dataframe, daily_dataframe

# Fetch and process data for all cities
hourly_dataframes = []
daily_dataframes = []

for city, (lat, lon) in cities.items():
    hourly_dataframe, daily_dataframe = fetch_and_process_weather_data(city, lat, lon)
    hourly_dataframes.append(hourly_dataframe)
    daily_dataframes.append(daily_dataframe)

# Concatenate all dataframes and save to CSV
hourly_final_dataframe = pd.concat(hourly_dataframes, ignore_index=True)
daily_final_dataframe = pd.concat(daily_dataframes, ignore_index=True)

hourly_final_dataframe.to_csv('hourly_weather_data.csv', index=False)
daily_final_dataframe.to_csv('daily_weather_data.csv', index=False)


Processing data for Delhi...
Coordinates 28.625°N 77.125°E
Elevation 221.0 m asl
Timezone None None
Timezone difference to GMT+0 0 s
Processing data for Mumbai...
Coordinates 19.125°N 72.875°E
Elevation 8.0 m asl
Timezone None None
Timezone difference to GMT+0 0 s
Processing data for Kolkata...
Coordinates 22.5°N 88.375°E
Elevation 5.0 m asl
Timezone None None
Timezone difference to GMT+0 0 s
Processing data for Bangalore...
Coordinates 13.0°N 77.625°E
Elevation 919.0 m asl
Timezone None None
Timezone difference to GMT+0 0 s
Processing data for Hyderabad...
Coordinates 17.375°N 78.5°E
Elevation 515.0 m asl
Timezone None None
Timezone difference to GMT+0 0 s
Processing data for Chennai...
Coordinates 13.0°N 80.125°E
Elevation 15.0 m asl
Timezone None None
Timezone difference to GMT+0 0 s
Processing data for Ahmedabad...
Coordinates 23.0°N 72.625°E
Elevation 53.0 m asl
Timezone None None
Timezone difference to GMT+0 0 s
Processing data for Surat...
Coordinates 21.125°N 72.875°E
Elevation